In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

# Cấu hình thiết bị chạy
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1 & 2. Tải tập dữ liệu huấn luyện (train=True) và tập xác thực (train=False)
data_folder = '~/data/FMNIST'
fmnist = datasets.FashionMNIST(data_folder, download=True, train=True)
tr_images = fmnist.data
tr_targets = fmnist.targets

val_fmnist = datasets.FashionMNIST(data_folder, download=True, train=False)
val_images = val_fmnist.data
val_targets = val_fmnist.targets

# 4. Tạo lớp Tìm nạp dữ liệu (FMNISTDataset) kèm chuẩn hóa pixel /255
class FMNISTDataset(Dataset):
    def __init__(self, x, y):
        x = x.float() / 255.0
        x = x.view(-1, 28 * 28)
        self.x, self.y = x, y
        
    def __getitem__(self, ix):
        x, y = self.x[ix], self.y[ix]
        return x.to(device), y.to(device)
        
    def __len__(self):
        return len(self.x)

In [ ]:
# Định nghĩa cấu trúc mạng và Trình tối ưu hóa Adam
def get_model():
    model = nn.Sequential(
        nn.Linear(28 * 28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=1e-2) # Chuyển sang dùng Adam Optimizer
    return model, loss_fn, optimizer

# Hàm huấn luyện một batch dữ liệu
def train_batch(x, y, model, opt, loss_fn):
    model.train()
    prediction = model(x)
    batch_loss = loss_fn(prediction, y)
    batch_loss.backward()
    opt.step()
    opt.zero_grad()
    return batch_loss.item()

# Hàm tính toán độ lỗi trên tập Validation (Tắt gradient)
@torch.no_grad()
def val_loss(x, y, model, loss_fn):
    model.eval()
    prediction = model(x)
    val_loss_value = loss_fn(prediction, y)
    return val_loss_value.item()

# Hàm tính toán độ chính xác Accuracy (Tắt gradient)
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    max_values, argmaxes = prediction.max(-1)
    is_correct = argmaxes == y
    return is_correct.cpu().numpy().tolist()

In [ ]:
# 5. Hàm get_data trả về cả DataLoader huấn luyện và xác thực
def get_data():
    train = FMNISTDataset(tr_images, tr_targets)
    trn_dl = DataLoader(train, batch_size=32, shuffle=True)
    
    val = FMNISTDataset(val_images, val_targets)
    # Tập val lấy batch_size bằng toàn bộ kích thước tập val, không cần shuffle
    val_dl = DataLoader(val, batch_size=len(val_images), shuffle=False)
    return trn_dl, val_dl

In [ ]:
# 7. Khởi tạo DataLoaders và Mô hình
trn_dl, val_dl = get_data()
model, loss_fn, optimizer = get_model()

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

# 8. Huấn luyện trong 5 Epochs
for epoch in range(5):
    print(f"Epoch đang chạy: {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # Lặp qua tập Huấn luyện (Cập nhật trọng số)
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # Tính Accuracy tập Huấn luyện sau Epoch
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # Lặp qua tập Xác thực (Chỉ đánh giá, không cập nhật trọng số)
    val_epoch_accuracies = []
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
        val_epoch_accuracies.extend(val_is_correct)
    val_epoch_accuracy = np.mean(val_epoch_accuracies)
    
    # Lưu kết quả lịch sử
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(val_epoch_accuracy)

In [ ]:
# 9. Trực quan hóa dữ liệu cải thiện
epochs = np.arange(5) + 1
plt.figure(figsize=(10, 10))

# Biểu đồ so sánh Loss (Train vs Validation)
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss when batch size is 32')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Biểu đồ so sánh Accuracy (Train vs Validation)
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy when batch size is 32')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()